## 前処理06: 投球軌道特徴量の追加

本ファイルは以下の処理を行うコードである。

1. `/workspace/datasets/statcast-customized-tmp/preprocess_05` のデータセットを読みだす。

2. `/workspace/datasets/statcast-customized-tmp/preprocess_01` から投球軌道に関する特徴量を取得し、以下の8カラムを追加する。
   - `vx0`, `vy0`, `vz0`: 投球の初速度ベクトル (ft/s)
   - `ax`, `ay`, `az`: 投球の加速度ベクトル (ft/s²)
   - `sz_top`, `sz_bot`: 打者のストライクゾーン上限・下限 (ft)

3. カラム追加後のデータを `/workspace/datasets/statcast-customized-tmp/preprocess_06` に保存する。

4. 保存後、`/workspace/datasets/statcast-customized/data` にコピーして運用に反映する。

### 前提
- `preprocess_01` と `preprocess_05` でファイル名・行数が一致していること（同一行が同一投球に対応）。
- これらのカラムは連続値特徴量として正規化され、モデルの `continuous_features` に追加される。

In [1]:
import os
import pandas as pd

In [2]:
raw_dir = '/workspace/datasets/statcast-customized-tmp/preprocess_01'
load_root_path = '/workspace/datasets/statcast-customized-tmp/preprocess_05/'
save_root_path = '/workspace/datasets/statcast-customized-tmp/preprocess_06/'

new_cols = ['vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot']

In [3]:
def get_parquet_file_path_list(root_path):
    file_path_list = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.parquet'):
                file_path_list.append(os.path.join(root, file))
    file_path_list.sort()
    return file_path_list

In [4]:
# 対象ファイル一覧を取得
parquet_file_path_list = get_parquet_file_path_list(load_root_path)
print(f'対象ファイル数: {len(parquet_file_path_list)}')
for f in parquet_file_path_list[:5]:
    print(f'  {f}')
print('  ...')

対象ファイル数: 72
  /workspace/datasets/statcast-customized-tmp/preprocess_05/statcast_2017_04.parquet
  /workspace/datasets/statcast-customized-tmp/preprocess_05/statcast_2017_05.parquet
  /workspace/datasets/statcast-customized-tmp/preprocess_05/statcast_2017_06.parquet
  /workspace/datasets/statcast-customized-tmp/preprocess_05/statcast_2017_07.parquet
  /workspace/datasets/statcast-customized-tmp/preprocess_05/statcast_2017_08.parquet
  ...


In [5]:
# 1ファイルで事前確認: カラムの存在とデータの整合性
test_path = parquet_file_path_list[0]
test_fname = os.path.basename(test_path)

df_src_test = pd.read_parquet(os.path.join(raw_dir, test_fname), columns=new_cols)
df_dst_test = pd.read_parquet(test_path)

print(f'ソースファイル (preprocess_01) 行数: {len(df_src_test)}')
print(f'入力ファイル (preprocess_05) 行数: {len(df_dst_test)}')
print(f'行数一致: {len(df_src_test) == len(df_dst_test)}')
print()
print('追加カラムの統計:')
display(df_src_test.describe())

ソースファイル (preprocess_01) 行数: 89094
入力ファイル (preprocess_05) 行数: 89094
行数一致: True

追加カラムの統計:


,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot
count,87836.000000,87836.000000,87836.000000,87836.000000,87836.000000,87836.000000,87836.000000,87836.000000
mean,2.690194,-128.390506,-4.609150,-3.401523,26.525679,-21.344228,3.638695,1.649534
std,6.060482,8.605027,3.047330,11.033289,3.988028,9.150497,0.245172,0.150076
min,-19.568353,-148.196839,-16.731449,-36.059338,8.842292,-72.189659,2.620000,0.850000
25%,-1.586064,-135.014301,-6.676893,-12.462392,23.542659,-27.537858,3.470000,1.540000
50%,4.430278,-130.379448,-4.816585,-4.063767,26.584120,-20.141070,3.690000,1.600000
75%,7.038305,-122.481519,-2.617642,4.977813,29.412775,-14.168976,3.840000,1.740000
max,19.428637,-84.479491,11.957131,32.500154,42.927690,10.518662,4.480000,2.430000


In [6]:
# 全ファイルにカラムを追加して preprocess_06 に保存
os.makedirs(save_root_path, exist_ok=True)

for parquet_path in parquet_file_path_list:
    fname = os.path.basename(parquet_path)
    raw_path = os.path.join(raw_dir, fname)

    if not os.path.exists(raw_path):
        print(f'SKIP (ソースなし): {fname}')
        continue

    df_raw = pd.read_parquet(raw_path, columns=new_cols)
    df = pd.read_parquet(parquet_path)

    if len(df_raw) != len(df):
        print(f'ERROR 行数不一致: {fname} (raw={len(df_raw)}, src={len(df)})')
        continue

    for col in new_cols:
        df[col] = df_raw[col].values

    save_path = os.path.join(save_root_path, fname)
    df.to_parquet(save_path, index=False)
    print(f'OK: {fname} ({len(df)} rows, +{len(new_cols)} cols)')

OK: statcast_2017_04.parquet (89094 rows, +8 cols)
OK: statcast_2017_05.parquet (103800 rows, +8 cols)
OK: statcast_2017_06.parquet (102555 rows, +8 cols)
OK: statcast_2017_07.parquet (96827 rows, +8 cols)
OK: statcast_2017_08.parquet (109454 rows, +8 cols)
OK: statcast_2017_09.parquet (105790 rows, +8 cols)
OK: statcast_2017_10.parquet (13793 rows, +8 cols)
OK: statcast_2017_11.parquet (277 rows, +8 cols)
OK: statcast_2018_03.parquet (10823 rows, +8 cols)
OK: statcast_2018_04.parquet (103761 rows, +8 cols)
OK: statcast_2018_05.parquet (111396 rows, +8 cols)
OK: statcast_2018_06.parquet (107721 rows, +8 cols)
OK: statcast_2018_07.parquet (103296 rows, +8 cols)
OK: statcast_2018_08.parquet (112622 rows, +8 cols)
OK: statcast_2018_09.parquet (107315 rows, +8 cols)
OK: statcast_2018_10.parquet (10208 rows, +8 cols)
OK: statcast_2019_03.parquet (27192 rows, +8 cols)
OK: statcast_2019_04.parquet (108003 rows, +8 cols)
OK: statcast_2019_05.parquet (116225 rows, +8 cols)
OK: statcast_2019_06.

In [7]:
# 結果の検証
saved_files = get_parquet_file_path_list(save_root_path)
print(f'保存ファイル数: {len(saved_files)}')

df_verify = pd.read_parquet(saved_files[0])
print(f'カラム数: {len(df_verify.columns)}')
print(f'カラム一覧: {list(df_verify.columns)}')
print()
print('追加カラムの欠損値:')
for col in new_cols:
    n_nan = df_verify[col].isna().sum()
    print(f'  {col}: {n_nan}/{len(df_verify)} ({n_nan/len(df_verify)*100:.2f}%)')

保存ファイル数: 72
カラム数: 31
カラム一覧: ['at_bat_id', 'swing_attempt', 'swing_result', 'bb_type', 'launch_speed', 'launch_angle', 'hit_distance_sc', 'p_throws', 'pitch_type', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'batter', 'stand', 'base_out_state', 'count_state', 'inning_clipped', 'is_inning_top', 'diff_score_clipped', 'pitch_number_clipped', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot']

追加カラムの欠損値:
  vx0: 1258/89094 (1.41%)
  vy0: 1258/89094 (1.41%)
  vz0: 1258/89094 (1.41%)
  ax: 1258/89094 (1.41%)
  ay: 1258/89094 (1.41%)
  az: 1258/89094 (1.41%)
  sz_top: 1258/89094 (1.41%)
  sz_bot: 1258/89094 (1.41%)


### 運用データへの反映

以下のコマンドで `preprocess_06` の parquet ファイルを `/workspace/datasets/statcast-customized/data` にコピーする。

```bash
cp /workspace/datasets/statcast-customized-tmp/preprocess_06/*.parquet /workspace/datasets/statcast-customized/data/
```

In [8]:
import shutil

final_data_dir = '/workspace/datasets/statcast-customized/data'
os.makedirs(final_data_dir, exist_ok=True)

for fpath in saved_files:
    fname = os.path.basename(fpath)
    dst = os.path.join(final_data_dir, fname)
    shutil.copy2(fpath, dst)

print(f'コピー完了: {len(saved_files)} ファイル → {final_data_dir}')

コピー完了: 72 ファイル → /workspace/datasets/statcast-customized/data
